# AGENTIC RAG OPTIMIZER
 ## This project builds an intelligent RAG system using agents to improve answer quality

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains.retrieval_qa.base import RetrievalQA
print("everything stable")

everything stable


In [6]:
from langchain_community.document_loaders import TextLoader
print("1.done")
from langchain.text_splitter import RecursiveCharacterTextSplitter
print("2.done")
from langchain_community.embeddings import HuggingFaceEmbeddings
print("3.done")
from langchain_community.vectorstores import FAISS
print("4.done")
from langchain.chains.retrieval_qa.base import RetrievalQA
print("5.done")

1.done
2.done
3.done
4.done
5.done


## 1-DATA INGESTION
### WE LOAD THE RAW DOCUMENTS AND PREPARE THEM FOR PROCESSING

In [10]:
loader = PyPDFLoader("Research_Paper_on_Artificial_Intelligence.pdf")
documents = loader.load()
print(documents)

[Document(page_content='www.ijecs.in \nInternational Journal Of Engineering And Computer Science \nVolume 12 Issue 02, February 2023, Page No.25654-25656 \nISSN: 2319-7242 DOI: 10.18535/ijecs/v11i02.4671 \n \n \n \n7  Ijecs 02 February Rajiv Gupta Research Paper on Artificial Intelligence \n \nPage | \n25654 \nResearch Paper on Artificial Intelligence \n \nRajiv Gupta \nChandigarh University, Chandigarh,  Haryana \n \n \nAbstract: This branch of computer science is concerned with making computers behave like humans. \nArtificial intelligence includes game playing, expert systems, neural networks, natural language, and robotics. \nCurrently, no computers exhibit full artificial intelligence (that is, are able to simulate human behavior). The \ngreatest advances have occurred in the field of games playing. The best computer chess programs are now \ncapable of beating humans. Today, the hottest area of artificial intelligence is neural networks, which are \nproving successful in a number 

## 2-TEXT CHUNKING
### THE DOCUMENTS ARE SPLIT INTO SMALLER CHUNKS TO IMPROVE RETRIEVAL ACCURACY

In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=100)
texts = text_splitter.split_documents(documents)
print("number of chunks:", len(texts))
print(texts[0].page_content)

number of chunks: 12
www.ijecs.in 
International Journal Of Engineering And Computer Science 
Volume 12 Issue 02, February 2023, Page No.25654-25656 
ISSN: 2319-7242 DOI: 10.18535/ijecs/v11i02.4671 
 
 
 
7  Ijecs 02 February Rajiv Gupta Research Paper on Artificial Intelligence 
 
Page | 
25654 
Research Paper on Artificial Intelligence 
 
Rajiv Gupta 
Chandigarh University, Chandigarh,  Haryana 
 
 
Abstract: This branch of computer science is concerned with making computers behave like humans. 
Artificial intelligence includes game playing, expert systems, neural networks, natural language, and robotics. 
Currently, no computers exhibit full artificial intelligence (that is, are able to simulate human behavior). The 
greatest advances have occurred in the field of games playing. The best computer chess programs are now 
capable of beating humans. Today, the hottest area of artificial intelligence is neural networks, which are


## 3-EMBEDDINGS
### TEXT CHUNKS ARE CONVERTED INTO VECTOR EMBEDDINGS FOR SEMANTIC SEARCH

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()
print(" embedding working")

C:\Users\RADHA\anaconda3\envs\rag_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2542.35it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 embedding working


## 4-VECTOR STORE(FAISS)
### WE STORE EMBEDDINGS IN FAISS FOR EFFICIENT SIMILARITY-BASED RETRIEVAL

In [16]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(texts, embeddings)

## 5-RETRIEVER
### RELEVANT CHUNKS ARE RETRIEVED BASED ON THE QUERY

In [17]:
retriever = vectorstore.as_retriever()

## 6-RESPONSE GENERATION
### THE RETRIEVED CONTEXT IS PASSED TO THE LLM TO GENERATE AN ANSWER .

In [18]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
def simple_llm(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
query = "what is this document about?"
docs = retriever.get_relevant_documents(query)
context = "".join([doc.page_content for doc in docs])
prompt = f""" Answer based on context:{context} 
question: {query}
"""
answer = simple_llm(prompt)
print(answer)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 2183.63it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
C:\Users\RADHA\anaconda3\envs\rag_clean\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Token indices sequence length is longer than the specified maximum sequence length for this model (842 > 512). Running this sequence through the model will result in indexing errors


Science/Tech


# AGENTIC /SMARTER RAG -AGENTIC OPTIMIZATION
### AN Agent evaluates the generated response and improves it if nedded .This step enhances accuracy and completeness


In [19]:
def multi_query_rag(queries, retriever, llm_func):
    answers = []
    for query in queries:
        docs = retriever.get_relevant_documents(query)
        context = " ".join([doc.page_content for doc in docs])
        prompt =  f""" Answer the questions based on the context below.
        if the context does not cotain the answer say "not mentioned":{context} 
        question: {query}
        """         
        answer = llm_func(prompt)
        answers.append({"query":query, "answer":answer})
    return answers

In [20]:
queries = ["what is the main contribution of this paper?",
          "which datasets are used",
          "what are the limitations mentioned?"]
answers = multi_query_rag(queries, retriever, simple_llm)
for item in answers:
    print(f"Query: {item['query']}")
    print(f"answer: {item['answer']}")
    print("-----")


Query: what is the main contribution of this paper?
answer: Artificial intelligence
-----
Query: which datasets are used
answer: not mentioned
-----
Query: what are the limitations mentioned?
answer: not mentioned
-----


In [21]:
# smarter multi query 
def smarter_multi_query_rag(queries, retriever, llm_func, keyword_filter=True, top_k=5):
    """
    queries : list of questions
    retriever: your faiss retriever
    llm_func: function to generate answers(simple_llm)
    ketword_filter: if True filter chunks containing keywords from query
    top_k :no of chunks retrieved
    """
    answers = []
    for query in queries:
        docs = retriever.get_relevant_documents(query)
        
        if keyword_filter:
            keywords = query.lower().split()
            filtered_docs = []
            for doc in docs:
                if any(kw in doc.page_content.lower() for kw in keywords):
                    filtered_docs.append(doc)
            if filtered_docs:
                docs = filtered_docs
        context = " ".join([doc.page_content for doc in docs])
        prompt =  f""" Answer the questions based on the context below.
        if the context does not cotain the answer say "not mentioned":{context} 
        question: {query}
        """         
        answer = llm_func(prompt)
        answers.append({"query":query, "answer":answer})
    return answers

In [22]:
queries = ["what is the main contribution of this paper?",
          "which datasets are used",
          "what are the limitations mentioned?"]
answers = smarter_multi_query_rag(queries, retriever, simple_llm, keyword_filter=True, top_k=5)
for item in answers:
    print(f"Query: {item['query']}")
    print(f"answer: {item['answer']}")
    print("-----")


Query: what is the main contribution of this paper?
answer: Artificial intelligence
-----
Query: which datasets are used
answer: CADIMA
-----
Query: what are the limitations mentioned?
answer: not mentioned
-----


In [23]:
def summarize_text(text, max_length=300):
    """summarize a long text chunk using flat t5"""
    inputs = tokenizer("summarize the following text:\n" + text,
                      return_tensors="pt",
                      truncation=True,
                      max_length=1024)
    outputs = model.generate(**inputs,
                            max_length=max_length)
    return tokenizer.decode(outputs[0],
                           skip_special_tokens=True)

In [24]:
summaries = []
for doc in texts:
    summary = summarize_text(doc.page_content)
    summaries.append(summary)
full_summary = "\n".join(summaries)
print("summary generated")

summary generated


In [25]:
def summarized_rag(queries, llm_func, summary_text):
    """
    queries :"""
    answers = []
    for query in queries:
        prompt = f""" Answer the questions based on the context below.
        if the context does not cotain the answer say "not mentioned" .
        summary:{summary_text} 
        question: {query}
        answer:
        """    
        answer = llm_func(prompt)
        answers.append({"query":query, "answer":answer})
    return answers
        

In [26]:
queries = ["what is the main contribution of this paper?",
          "which datasets are used",
          "what are the limitations mentioned?"]
answers = multi_query_rag(queries, retriever, simple_llm)
for item in answers:
    print(f"Query: {item['query']}")
    print(f"answer: {item['answer']}")
    print("-----")

Query: what is the main contribution of this paper?
answer: Artificial intelligence
-----
Query: which datasets are used
answer: not mentioned
-----
Query: what are the limitations mentioned?
answer: not mentioned
-----


# fully Agentic RAG Layer :Autonomous and smarter

In [27]:
from langchain.prompts import PromptTemplate


In [28]:
agentic_prompt = PromptTemplate(input_variables=["context", "query"],
                               template="""
                               Answer the following questions based on the context below.
                               if the context does not contain the answer, say "not mentioned".
                               context:{context}
                               question:{query}
                               Answer in JSON format with the fields:
                               -"query": the question
                               -"answer": the answer text
                               -"source_pages": list of page numbers(if available)
                               
                               Answer:
                               """)

In [29]:
def agentic_rag(queries, retriever, llm_func, top_k=5, summarize=False):
    """ queries: list of questions
        retriever: FIASS retriever
        llm_func: no of chunks to fetch 
        summarize: wheather to summarize retriver chunks first
        """
    results = []
    for query  in queries:
        #Retriever top_k chunks
        docs = retriever.get_relevant_documents(query)[:top_k]
        # optional summarize retrieved chunks
        if summarize:
            context = "\n".join([summarize_text(doc.page_content) for doc in docs])
        else:
            context = "\n".join([doc.page_content for doc in docs])

        #build prompt using a agentic template
        prompt = agentic_prompt.format(context=context, query=query)

        #generate answer
        answer_text = llm_func(prompt)
        #optional:parse JSON manually if needed
        results.append({"query":query, "answer": answer_text})
    return results
    


In [30]:
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

In [31]:

queries = ["what is the main contribution of this paper?",
          "which datasets are used",
          "what are the limitations mentioned?"]
answers = agentic_rag(queries, retriever, simple_llm, top_k=5, summarize=True)
for item in answers:
    print(f"Query: {item['query']}")
    print(f"answer: {item['answer']}")
    print("-----")

Query: what is the main contribution of this paper?
answer: not mentioned
-----
Query: which datasets are used
answer: not mentioned
-----
Query: what are the limitations mentioned?
answer: not mentioned
-----


# Reflective Agent

In [32]:
from transformers import pipeline


In [34]:
llm = pipeline("text-generation", model="distilgpt2")

Loading weights: 100%|███████████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 1700.04it/s]


In [41]:
# CLEAN OUTPUT
def clean_output(text, prompt):
   return text.replace(prompt,"").strip()

In [61]:
def reflective_agentic_rag(question):
    docs = retriever.get_relevant_documents(question)

    if not docs:
        return "not found in context "

    context = " ".join([doc.page_content for doc in docs])
    #if question is about summary
    if"about" in question.lower() or "summary" in question.lower():
        prompt = f""" 
        summarize the following context in 2-3 lines:

        {context}
        """

        result = llm(prompt, max_length=100)[0]["generated_text"]
        answer = clean_output(result, prompt)
        return answer

        keyword = ["dataset", "data set", "trained on", "used", "model"]
        lines = context.split(".")
        for line in lines:
            for keyword in keywords:
                if keyword in line.lower():
                    return line.strip()
        return "not found in context"
    
    
   
     




In [62]:
def score_answer(question):
    docs = retriever.get_relevant_documents(question)

    if not docs:
        return"no context", "Score:0"

    context = "".join([doc.page_content for doc in docs])

    answer = reflective_agentic_rag(question)
    #simple rule based scoring
    if answer.lower() in context.lower():
        score = "score: 8/10"
    elif len(answer.strip())<10:
        score = "score:3/10"
    else:
        score = "score:5/10"
    return answer, score

    
    

    

In [63]:
#FINAL DEMO
def final_demo(question):
    answer, score = score_answer(question)

    

    print("question:", question)
    print("\nFinal Answer:", answer)
    print("\nScore:\n", score)

In [64]:
reflective_agentic_rag("what is this context about?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'ids. One can predict the future of the AI, but one can not predict the future of the AI, and the next time you want to predict the future of AI, you can guess that the future of AI will be very different than the other. The future of the AI will be very different than the future of the AI, but one can not predict the future of the AI.\nIn the present context, we have the best guess at some point of the future of AI, but there is at least one option that may need to be taken to solve the problem. In the present context, we have the best guess at some point of the future of AI, but there is at least one option that may need to be taken to solve the problem. In the present context, we have the best guess at some point of the future of AI, but there is at least one option that may need to be taken to solve the problem. In the present context, we have the best guess at some point of the future of AI, but there is at least one option that may need to be taken to solve the problem. In the pr

In [52]:
def debug_retrieval(question):
    docs = retriever.get_relevant_documents(question)

    print("Retrived documents:\n")
    for i,doc in enumerate(docs):
        print(f"Doc{i+1}:\n", doc.page_content[:500],"\n")

In [53]:
debug_retrieval("which dataset used?")

Retrived documents:

Doc1:
 www.ijecs.in 
International Journal Of Engineering And Computer Science 
Volume 12 Issue 02, February 2023, Page No.25654-25656 
ISSN: 2319-7242 DOI: 10.18535/ijecs/v11i02.4671 
 
 
 
7  Ijecs 02 February Rajiv Gupta Research Paper on Artificial Intelligence 
 
Page | 
25654 
Research Paper on Artificial Intelligence 
 
Rajiv Gupta 
Chandigarh University, Chandigarh,  Haryana 
 
 
Abstract: This branch of computer science is concerned with making computers behave like humans. 
Artificial intelli 

Doc2:
 Ijecs 02 February Rajiv Gupta Research Paper on Artificial Intelligence 
 Page | 
25655 
that the hydraulic fracturing is carried out at the 
level of -210 m to -434 m, with a hole depth of 
224 m, at -210 m horizontal drilling downward 
vertical drilling. Therefore, according to the 
obtained final mining range at the level of -434 m 
is measured, and it is finally determined that the 
range of parallel ore body strike of 850 m and 
vertical ore body strike

In [67]:
def reflective_agentic_rag(question):
    docs = retriever.get_relevant_documents(question)

    if not docs:
        return "not found in context "

    context = docs[0].page_content
    #try to extract documents
    lines = context.split(".")
    for line in lines:
        if any(word in line.lower() for word in ["dataset", "trained", "data"]):
            return f"Answer:{line.strip()}\n\nSource:{context[:200]}..."
            # fall back
    return f"Context found but exact answer unclear.\n\nsource:{context[:200]}.."

In [68]:
def score_answer(question):
    docs = retriever.get_relevant_documents(question)

    if not docs:
        return"no context", "Score:0"

    context = "".join([doc.page_content for doc in docs])

    answer = reflective_agentic_rag(question)
    #simple rule based scoring
    if answer.lower() in context.lower():
        score = "score: 8/10"
    elif len(answer.strip())<10:
        score = "score:3/10"
    else:
        score = "score:5/10"
    return answer, score

    

In [69]:
#FINAL DEMO
def final_demo(question):
    answer, score = score_answer(question)

    

    print("question:", question)
    print("\nFinal Answer:", answer)
    print("\nScore:\n", score)

# FINAL OUTPUT
### The system successfully retrieves relevant context using the RAG pipeline. However, in somecases the generated answer may be unclear or incomplete

In [70]:
final_demo("which dataset is used?")

question: which dataset is used?

Final Answer: Context found but exact answer unclear.

source:Ijecs 02 February Rajiv Gupta Research Paper on Artificial Intelligence 
 Page | 
25655 
that the hydraulic fracturing is carried out at the 
level of -210 m to -434 m, with a hole depth of 
224 m, at..

Score:
 score:5/10


## LIMITATIONS

### -Retrieved context is relevant but not always fully utilized by the LLM 
### -Answer generation may be vague for complex queries 
### -No strong evaluation matric is applied yet

## FUTURE IMPROVEMENTS

### -Improve prompt Engineering for better answer generaion
### -Introduce evaluation metric(e.g answer relevance scoring)
### - Use strong LLM or fine-tuned model
### - Implement multi-agent feedback loop for refinement